<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test_H1_HeatKernel_FractalMemory_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG_Test_H1_HeatKernel_FractalMemory_Audit

Continuation of `ROSG_Test_H1_DeltaTau_Blind_LogPeriodic_Residual_SELFCONTAINED.ipynb`.

This notebook tests whether the Z-coupled H1 channel retains a heat-kernel fractal memory:

\[
P_i(u,Z)=\operatorname{Tr}(e^{-uL_{i,Z}})
\]

The H1 branch receives no log-periodic injection. The audit searches blindly for periodicity in `log(u)` after power-law detrending of `log P`. H0/static/nonlattice/smooth-spectrum/dropout and residual surrogates are included. A positive control is injected only to verify detector sensitivity.


In [1]:
# ============================================
# ROSG_Test_H1_HeatKernel_FractalMemory_Audit
# Continuity with ROSG_Test_H1_DeltaTau_Blind_LogPeriodic_Residual_SELFCONTAINED.ipynb
#
# Question:
#   If Weyl-counting / residual-Z tests do not provide a specific spontaneous DSI claim,
#   does the heat-kernel trace P(u; Z)=Tr(exp(-u L_{i,Z})) retain a specific log-periodic
#   "fractal memory" in log diffusion time, beyond H0/static/surrogate controls?
#
# Discipline:
#   - H1 is Z-coupled through DeltaTau: Z = ln(DeltaTau_sun / t_P).
#   - No log-periodic signal is injected in the H1 test branch.
#   - Periodicities in log(u) are estimated blindly after power-law detrending of log P.
#   - H0, static-H1, smooth-spectrum, nonlattice-Z, residual phase/circular surrogates included.
#   - Positive control is detector-only and explicitly injected.
# ============================================

import json
import math
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


@dataclass
class HeatKernelMemoryConfig:
    levels: Tuple[int, ...] = (1, 2, 3, 4, 5)
    # DeltaTau / Z resolution coupling.
    tP_s: float = 5.391246446661944e-44
    Z_min: float = 0.0
    Z_max: float = 6.5
    n_Z: int = 121
    Zc0: float = 1.05
    delta_Z_level: float = math.log(2.0)
    activation_width: float = 0.32
    g_min: float = 0.0
    g_max: float = 0.35
    dropout_max: float = 0.45
    dropout_width: float = 0.45
    # Heat-kernel diffusion-time grid.
    logu_min: float = -2.0
    logu_max: float = 5.0
    n_u: int = 72
    # Z slices relative to nominal activation center.
    z_slice_offsets: Tuple[float, ...] = (-0.9, 0.0, 0.9)
    z_slice_labels: Tuple[str, ...] = ("pre_activation", "activation_center", "post_saturation")
    # Blind logu period scan.
    period_min_logu: float = 0.45
    period_max_logu: float = 2.80
    n_omega: int = 140
    n_permutations: int = 60
    fap_threshold: float = 0.05
    min_power_threshold: float = 0.10
    coherence_cv_threshold: float = 0.25
    min_levels_for_coherent_claim: int = 3
    power_margin_threshold: float = 1.25
    # Detrending window and model.
    trim_fraction_each_side: float = 0.12
    # Positive control detector check only.
    positive_control_amplitude_logP: float = 0.040
    positive_control_period_logu: float = math.log(4.0)
    positive_control_phase: float = 0.25
    # Output.
    out_dir: str = "/mnt/data/ROSG_Test_H1_HeatKernel_FractalMemory_Audit_out"
    random_seed: int = 20260707


def grid_torus_eigenvalues(L: int) -> np.ndarray:
    """Combinatorial Laplacian eigenvalues on an LxL periodic square grid."""
    ks = np.arange(L, dtype=float)
    vals1 = 2.0 - 2.0 * np.cos(2.0 * np.pi * ks / L)
    lam = vals1[:, None] + vals1[None, :]
    return np.sort(lam.ravel())


def h1_spectrum_from_grid(lam_plane: np.ndarray, g: float) -> np.ndarray:
    """Two-layer H1 spectrum: symmetric planar branch + antisymmetric fiber-shifted branch."""
    return np.sort(np.concatenate([lam_plane, lam_plane + 2.0 * max(g, 0.0)]))


def positive_eigs(eigs: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    e = np.asarray(eigs, float)
    return e[e > eps]


def heat_trace(eigs_pos: np.ndarray, u: np.ndarray) -> np.ndarray:
    # Mean over positive modes to avoid zero-mode saturation dominating large u.
    e = np.asarray(eigs_pos, float)
    u = np.asarray(u, float)
    if len(e) == 0:
        return np.zeros_like(u)
    # Chunked for memory safety.
    out = np.zeros_like(u, dtype=float)
    chunk = 4096
    for start in range(0, len(e), chunk):
        ee = e[start:start+chunk]
        out += np.exp(-np.outer(u, ee)).sum(axis=1)
    return out / float(len(e))


def sigmoid_activation(Z: np.ndarray, zc: float, width: float, g_min: float, g_max: float) -> np.ndarray:
    return g_min + (g_max - g_min) * 0.5 * (1.0 + np.tanh((Z - zc) / max(width, 1e-12)))


def dropout_profile(Z: np.ndarray, zc: float, cfg: HeatKernelMemoryConfig) -> np.ndarray:
    # High dropout before activation, low dropout after stabilization.
    return cfg.dropout_max * 0.5 * (1.0 - np.tanh((Z - zc) / max(cfg.dropout_width, 1e-12)))


def level_center(cfg: HeatKernelMemoryConfig, i: int, nonlattice: bool = False) -> float:
    if nonlattice:
        return cfg.Zc0 + (i - 1) * cfg.delta_Z_level * math.sqrt(2.0) + 0.06 * math.sin(1.31 * i)
    return cfg.Zc0 + (i - 1) * cfg.delta_Z_level


def spectral_dimension_from_logP(logu: np.ndarray, logP: np.ndarray, trim: float) -> Tuple[float, float, float]:
    n = len(logu)
    a = int(max(0, math.floor(trim * n)))
    b = int(min(n, math.ceil((1.0 - trim) * n)))
    x = logu[a:b]
    y = logP[a:b]
    A = np.column_stack([np.ones_like(x), x])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    intercept, slope = coef
    ds_est = float(max(0.0, -2.0 * slope))
    return ds_est, float(intercept), float(slope)


def detrend_log_heat(logu: np.ndarray, logP: np.ndarray, cfg: HeatKernelMemoryConfig) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    # Power-law detrending in log-log: log P = a - ds/2 * log u + residual.
    ds_est, intercept, slope = spectral_dimension_from_logP(logu, logP, cfg.trim_fraction_each_side)
    baseline = intercept + slope * logu
    resid = logP - baseline
    return baseline, resid, {"ds_est_from_slope": ds_est, "linear_intercept": intercept, "linear_slope": slope}


_PERIODIC_BASIS_CACHE = {}

def _basis_key(x: np.ndarray, omegas: np.ndarray):
    return (len(x), float(x[0]), float(x[-1]), len(omegas), float(omegas[0]), float(omegas[-1]))

def precompute_orth_basis(x: np.ndarray, omegas: np.ndarray) -> np.ndarray:
    key = _basis_key(x, omegas)
    if key in _PERIODIC_BASIS_CACHE:
        return _PERIODIC_BASIS_CACHE[key]
    x = np.asarray(x, float)
    Qs = np.zeros((len(omegas), len(x), 2), dtype=float)
    for idx, w in enumerate(omegas):
        X = np.column_stack([np.cos(w*x), np.sin(w*x)])
        X = X - X.mean(axis=0, keepdims=True)
        # QR gives an orthonormal basis for the sinusoidal subspace after intercept removal.
        Q, _ = np.linalg.qr(X)
        Qs[idx, :, :] = Q[:, :2]
    _PERIODIC_BASIS_CACHE[key] = Qs
    return Qs

def powers_for_residual(x: np.ndarray, r: np.ndarray, omegas: np.ndarray) -> np.ndarray:
    x = np.asarray(x, float)
    r = np.asarray(r, float)
    r = r - np.mean(r)
    ss_tot = float(np.sum(r ** 2))
    if ss_tot <= 1e-20:
        return np.zeros_like(omegas)
    Qs = precompute_orth_basis(x, omegas)
    # projections: omega x 2
    proj = np.einsum('wnd,n->wd', Qs, r)
    return np.clip(np.sum(proj*proj, axis=1) / ss_tot, 0.0, 1.0)

def fit_at_omega(x: np.ndarray, r: np.ndarray, omega: float) -> Tuple[float, float, float]:
    X = np.column_stack([np.cos(omega * x), np.sin(omega * x), np.ones_like(x)])
    beta, *_ = np.linalg.lstsq(X, r, rcond=None)
    a, b, c = beta
    amp = float(math.sqrt(a*a + b*b))
    phase = float(math.atan2(-b, a))
    return amp, phase, float(c)


def scan_blind_periodicity(x: np.ndarray, resid: np.ndarray, cfg: HeatKernelMemoryConfig, rng: np.random.Generator) -> Dict[str, float]:
    periods = np.linspace(cfg.period_min_logu, cfg.period_max_logu, cfg.n_omega)
    omegas = 2.0 * np.pi / periods
    r = np.asarray(resid, float)
    sd = float(np.std(r))
    if sd <= 1e-14:
        return {
            "best_period_logu": float("nan"), "best_omega": float("nan"),
            "best_power": 0.0, "best_amplitude_std": 0.0, "best_phase": float("nan"),
            "best_offset_std": 0.0, "fap_permutation": 1.0,
            "null_max_power_median": 0.0, "null_max_power_q95": 0.0,
        }
    r_std = (r - np.mean(r)) / sd
    powers = powers_for_residual(x, r_std, omegas)
    j = int(np.argmax(powers))
    amp, phase, offset = fit_at_omega(x, r_std, float(omegas[j]))
    null_max = np.zeros(cfg.n_permutations, dtype=float)
    for k in range(cfg.n_permutations):
        rr = rng.permutation(r_std)
        null_max[k] = float(np.max(powers_for_residual(x, rr, omegas)))
    fap = (1.0 + float(np.sum(null_max >= powers[j]))) / (cfg.n_permutations + 1.0)
    return {
        "best_period_logu": float(periods[j]),
        "best_omega": float(omegas[j]),
        "best_power": float(powers[j]),
        "best_amplitude_std": float(amp),
        "best_phase": float(phase),
        "best_offset_std": float(offset),
        "fap_permutation": float(fap),
        "null_max_power_median": float(np.median(null_max)),
        "null_max_power_q95": float(np.quantile(null_max, 0.95)),
    }


def smooth_spectrum_surrogate(n_modes: int, lam_min: float, lam_max: float, ds_like: float = 2.0) -> np.ndarray:
    # Build a monotone Weyl-like spectrum without discrete degeneracy structure.
    # N(lambda) ~ lambda^(ds/2) => lambda_k ~ k^(2/ds).
    k = np.arange(1, n_modes + 1, dtype=float)
    p = max(2.0 / max(ds_like, 0.2), 0.1)
    x = (k / float(n_modes)) ** p
    return np.sort(lam_min + (lam_max - lam_min) * x)


def compute_curve_records_for_case(cfg: HeatKernelMemoryConfig, case: str, rng: np.random.Generator) -> Tuple[List[Dict], List[Dict]]:
    u = np.exp(np.linspace(cfg.logu_min, cfg.logu_max, cfg.n_u))
    logu = np.log(u)
    curves, periodic_rows = [], []
    Z_grid = np.linspace(cfg.Z_min, cfg.Z_max, cfg.n_Z)
    for i in cfg.levels:
        L = 2 ** (i + 1)
        lam_plane_all = grid_torus_eigenvalues(L)
        lam_plane_pos = positive_eigs(lam_plane_all)
        zc_lattice = level_center(cfg, i, nonlattice=False)
        zc_nonlattice = level_center(cfg, i, nonlattice=True)
        # Z slices are centered around the lattice nominal center, then clipped to scan range.
        for label, off in zip(cfg.z_slice_labels, cfg.z_slice_offsets):
            Z_slice = float(np.clip(zc_lattice + off, cfg.Z_min, cfg.Z_max))
            if case == "H0_planar":
                eigs = lam_plane_pos
                g_eff = 0.0
            elif case == "H1_Z_coupled":
                g_eff = float(sigmoid_activation(np.array([Z_slice]), zc_lattice, cfg.activation_width, cfg.g_min, cfg.g_max)[0])
                eigs = positive_eigs(h1_spectrum_from_grid(lam_plane_all, g_eff))
            elif case == "CTRL_H1_static_full_channel":
                g_eff = cfg.g_max
                eigs = positive_eigs(h1_spectrum_from_grid(lam_plane_all, g_eff))
            elif case == "CTRL_nonlattice_Z_centers":
                g_eff = float(sigmoid_activation(np.array([Z_slice]), zc_nonlattice, cfg.activation_width, cfg.g_min, cfg.g_max)[0])
                eigs = positive_eigs(h1_spectrum_from_grid(lam_plane_all, g_eff))
            elif case == "CTRL_smooth_Weyl_spectrum_matched_count":
                # Match mode count and range to H1 at same g, but erase discrete grid degeneracies.
                g_eff = float(sigmoid_activation(np.array([Z_slice]), zc_lattice, cfg.activation_width, cfg.g_min, cfg.g_max)[0])
                e_h1 = positive_eigs(h1_spectrum_from_grid(lam_plane_all, g_eff))
                eigs = smooth_spectrum_surrogate(len(e_h1), float(np.min(e_h1)), float(np.max(e_h1)), ds_like=2.0)
            elif case == "CTRL_dropout_Z_effective_shift":
                g_smooth = float(sigmoid_activation(np.array([Z_slice]), zc_lattice, cfg.activation_width, cfg.g_min, cfg.g_max)[0])
                pdrop = float(dropout_profile(np.array([Z_slice]), zc_lattice, cfg)[0])
                # Deterministic effective dropout: reduce fiber conductance by retained fraction.
                g_eff = g_smooth * (1.0 - pdrop)
                eigs = positive_eigs(h1_spectrum_from_grid(lam_plane_all, g_eff))
            else:
                raise ValueError(f"Unknown case {case}")
            P = heat_trace(eigs, u)
            logP = np.log(np.maximum(P, 1e-300))
            if case == "POS_injected_logu_periodic_detector_check":
                # This branch not used here; produced separately.
                pass
            baseline, resid, params = detrend_log_heat(logu, logP, cfg)
            periodic = scan_blind_periodicity(logu, resid, cfg, rng)
            level_pass = bool(periodic["fap_permutation"] <= cfg.fap_threshold and periodic["best_power"] >= cfg.min_power_threshold)
            periodic_rows.append({
                "case": case,
                "target_i": int(i),
                "grid_L": int(L),
                "Z_slice_label": label,
                "Z_slice": Z_slice,
                "Zc_lattice": zc_lattice,
                "Zc_nonlattice": zc_nonlattice,
                "g_eff": float(g_eff),
                "n_positive_modes": int(len(eigs)),
                **params,
                **periodic,
                "level_logu_periodicity_pass": level_pass,
                "expected_grid_log_period_ln4": float(math.log(4.0)),
                "abs_period_minus_ln4": float(abs(periodic["best_period_logu"] - math.log(4.0))) if np.isfinite(periodic["best_period_logu"]) else float("nan"),
                "residual_std_logP": float(np.std(resid)),
                "residual_ptp_logP": float(np.ptp(resid)),
            })
            sd = max(float(np.std(resid)), 1e-12)
            for uu, lu, pp, lp, bb, rr in zip(u, logu, P, logP, baseline, resid):
                curves.append({
                    "case": case,
                    "target_i": int(i),
                    "grid_L": int(L),
                    "Z_slice_label": label,
                    "Z_slice": Z_slice,
                    "u": float(uu),
                    "logu": float(lu),
                    "P_heat_trace": float(pp),
                    "logP": float(lp),
                    "logP_powerlaw_baseline": float(bb),
                    "heat_residual_logP_raw": float(rr),
                    "heat_residual_logP_std": float((rr - np.mean(resid)) / sd),
                    "g_eff": float(g_eff),
                })
    return curves, periodic_rows


def add_positive_control_from_h1(curves_df: pd.DataFrame, cfg: HeatKernelMemoryConfig, rng: np.random.Generator) -> Tuple[pd.DataFrame, pd.DataFrame]:
    pos_curves, pos_periods = [], []
    omega = 2.0 * math.pi / cfg.positive_control_period_logu
    for (i, label), d0 in curves_df[(curves_df["case"] == "H1_Z_coupled")].groupby(["target_i", "Z_slice_label"]):
        d = d0.sort_values("logu").copy()
        logu = d["logu"].to_numpy(float)
        logP = d["logP"].to_numpy(float)
        phase = cfg.positive_control_phase + 0.13 * (int(i) - 1)
        logP_pos = logP + cfg.positive_control_amplitude_logP * np.cos(omega * logu + phase)
        baseline, resid, params = detrend_log_heat(logu, logP_pos, cfg)
        periodic = scan_blind_periodicity(logu, resid, cfg, rng)
        level_pass = bool(periodic["fap_permutation"] <= cfg.fap_threshold and periodic["best_power"] >= cfg.min_power_threshold)
        pos_periods.append({
            "case": "POS_injected_logu_periodic_detector_check",
            "target_i": int(i),
            "grid_L": int(d["grid_L"].iloc[0]),
            "Z_slice_label": label,
            "Z_slice": float(d["Z_slice"].iloc[0]),
            "Zc_lattice": float("nan"),
            "Zc_nonlattice": float("nan"),
            "g_eff": float(d["g_eff"].iloc[0]),
            "n_positive_modes": int(len(d)),
            **params,
            **periodic,
            "level_logu_periodicity_pass": level_pass,
            "expected_grid_log_period_ln4": float(math.log(4.0)),
            "abs_period_minus_ln4": float(abs(periodic["best_period_logu"] - math.log(4.0))) if np.isfinite(periodic["best_period_logu"]) else float("nan"),
            "residual_std_logP": float(np.std(resid)),
            "residual_ptp_logP": float(np.ptp(resid)),
        })
        sd = max(float(np.std(resid)), 1e-12)
        for row, lp, bb, rr in zip(d.to_dict("records"), logP_pos, baseline, resid):
            row = dict(row)
            row["case"] = "POS_injected_logu_periodic_detector_check"
            row["logP"] = float(lp)
            row["logP_powerlaw_baseline"] = float(bb)
            row["heat_residual_logP_raw"] = float(rr)
            row["heat_residual_logP_std"] = float((rr - np.mean(resid)) / sd)
            pos_curves.append(row)
    return pd.DataFrame(pos_curves), pd.DataFrame(pos_periods)


def make_residual_surrogates(curves_df: pd.DataFrame, period_df: pd.DataFrame, cfg: HeatKernelMemoryConfig, rng: np.random.Generator) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Build phase/circular residual surrogates from H1 residuals only, preserving logu grid and baseline.
    s_curves, s_periods = [], []
    h1 = curves_df[curves_df["case"] == "H1_Z_coupled"].copy()
    for surrogate_name in ["CTRL_phase_surrogate_heat_residual", "CTRL_circular_shift_heat_residual"]:
        for (i, label), d0 in h1.groupby(["target_i", "Z_slice_label"]):
            d = d0.sort_values("logu").copy()
            logu = d["logu"].to_numpy(float)
            baseline = d["logP_powerlaw_baseline"].to_numpy(float)
            resid = d["heat_residual_logP_raw"].to_numpy(float)
            if surrogate_name == "CTRL_phase_surrogate_heat_residual":
                # Randomize Fourier phases while preserving amplitude spectrum.
                r = resid - np.mean(resid)
                fft = np.fft.rfft(r)
                amp = np.abs(fft)
                phases = rng.uniform(0, 2*np.pi, size=len(fft))
                phases[0] = 0.0
                if len(phases) > 1:
                    phases[-1] = 0.0
                surr = np.fft.irfft(amp * np.exp(1j * phases), n=len(r))
                surr = surr + np.mean(resid)
            else:
                shift = int(rng.integers(5, max(6, len(resid)-5)))
                surr = np.roll(resid, shift)
            logP_surr = baseline + surr
            base2, resid2, params = detrend_log_heat(logu, logP_surr, cfg)
            periodic = scan_blind_periodicity(logu, resid2, cfg, rng)
            level_pass = bool(periodic["fap_permutation"] <= cfg.fap_threshold and periodic["best_power"] >= cfg.min_power_threshold)
            s_periods.append({
                "case": surrogate_name,
                "target_i": int(i),
                "grid_L": int(d["grid_L"].iloc[0]),
                "Z_slice_label": label,
                "Z_slice": float(d["Z_slice"].iloc[0]),
                "Zc_lattice": float("nan"),
                "Zc_nonlattice": float("nan"),
                "g_eff": float(d["g_eff"].iloc[0]),
                "n_positive_modes": int(len(d)),
                **params,
                **periodic,
                "level_logu_periodicity_pass": level_pass,
                "expected_grid_log_period_ln4": float(math.log(4.0)),
                "abs_period_minus_ln4": float(abs(periodic["best_period_logu"] - math.log(4.0))) if np.isfinite(periodic["best_period_logu"]) else float("nan"),
                "residual_std_logP": float(np.std(resid2)),
                "residual_ptp_logP": float(np.ptp(resid2)),
            })
            sd = max(float(np.std(resid2)), 1e-12)
            for row, lp, bb, rr in zip(d.to_dict("records"), logP_surr, base2, resid2):
                row = dict(row)
                row["case"] = surrogate_name
                row["logP"] = float(lp)
                row["logP_powerlaw_baseline"] = float(bb)
                row["heat_residual_logP_raw"] = float(rr)
                row["heat_residual_logP_std"] = float((rr - np.mean(resid2)) / sd)
                s_curves.append(row)
    return pd.DataFrame(s_curves), pd.DataFrame(s_periods)


def summarize_periodicity(period_df: pd.DataFrame, cfg: HeatKernelMemoryConfig) -> pd.DataFrame:
    rows = []
    for (case, label), d in period_df.groupby(["case", "Z_slice_label"]):
        d = d[d["target_i"].isin(cfg.levels)].copy()
        passed = d[d["level_logu_periodicity_pass"]]
        periods = passed["best_period_logu"].to_numpy(float)
        coherent_count = int(len(passed))
        period_cv = float(np.std(periods) / max(abs(np.mean(periods)), 1e-12)) if len(periods) >= 2 else float("inf")
        coherent = bool(coherent_count >= cfg.min_levels_for_coherent_claim and period_cv <= cfg.coherence_cv_threshold)
        rows.append({
            "case": case,
            "Z_slice_label": label,
            "n_levels": int(len(d)),
            "n_level_periodicity_pass": coherent_count,
            "coherent_multilevel_periodicity": coherent,
            "median_best_power": float(d["best_power"].median()),
            "max_best_power": float(d["best_power"].max()),
            "median_fap": float(d["fap_permutation"].median()),
            "min_fap": float(d["fap_permutation"].min()),
            "median_best_period_logu": float(d["best_period_logu"].median()),
            "period_cv_among_passed": period_cv,
            "median_abs_period_minus_ln4": float(d["abs_period_minus_ln4"].median()),
            "median_ds_est": float(d["ds_est_from_slope"].median()),
            "median_residual_ptp_logP": float(d["residual_ptp_logP"].median()),
        })
    return pd.DataFrame(rows)


def global_verdict(summary_df: pd.DataFrame, cfg: HeatKernelMemoryConfig) -> Dict:
    h1_rows = summary_df[(summary_df["case"] == "H1_Z_coupled") & (summary_df["Z_slice_label"].isin(cfg.z_slice_labels))]
    # Prioritize activation_center and post_saturation as meaningful heat-kernel channel states.
    h1_meaningful = h1_rows[h1_rows["Z_slice_label"].isin(["activation_center", "post_saturation"])]
    h1_pass = bool(h1_meaningful["coherent_multilevel_periodicity"].any()) if len(h1_meaningful) else False
    h1_power = float(h1_meaningful["median_best_power"].max()) if len(h1_meaningful) else 0.0
    null_cases = [
        "H0_planar",
        "CTRL_H1_static_full_channel",
        "CTRL_nonlattice_Z_centers",
        "CTRL_smooth_Weyl_spectrum_matched_count",
        "CTRL_dropout_Z_effective_shift",
        "CTRL_phase_surrogate_heat_residual",
        "CTRL_circular_shift_heat_residual",
    ]
    null_df = summary_df[summary_df["case"].isin(null_cases) & summary_df["Z_slice_label"].isin(["activation_center", "post_saturation"])]
    null_coherent = null_df[null_df["coherent_multilevel_periodicity"]]
    null_controls_rejected = bool(len(null_coherent) == 0)
    max_null_power = float(null_df["median_best_power"].max()) if len(null_df) else 0.0
    margin = float(h1_power / max(max_null_power, 1e-12))
    power_margin_pass = bool(margin >= cfg.power_margin_threshold)
    pos_df = summary_df[summary_df["case"] == "POS_injected_logu_periodic_detector_check"]
    positive_ok = bool(pos_df["coherent_multilevel_periodicity"].any()) if len(pos_df) else False
    if h1_pass and null_controls_rejected and power_margin_pass:
        verdict = "H1_heatkernel_fractal_memory_candidate_specific_controls_rejected"
        claim = True
    elif h1_pass and not null_controls_rejected:
        verdict = "H1_heatkernel_logu_periodicity_not_specific_controls_or_surrogates_pass"
        claim = False
    elif not h1_pass:
        verdict = "H1_heatkernel_no_coherent_fractal_memory_detected"
        claim = False
    else:
        verdict = "H1_heatkernel_inconclusive"
        claim = False
    return {
        "heatkernel_fractal_memory_verdict": verdict,
        "levels_tested": list(cfg.levels),
        "Z_definition": "Z = ln(DeltaTau_sun / t_P)",
        "observable": "P_i(u,Z)=mean_positive_modes exp(-u lambda), residual of log P after power-law detrending in log u",
        "h1_meaningful_slices_coherent_pass": h1_pass,
        "null_controls_rejected": null_controls_rejected,
        "null_controls_or_surrogates_coherent": sorted(null_coherent["case"].unique().tolist()) if len(null_coherent) else [],
        "h1_power_margin_over_max_null": margin,
        "power_margin_pass": power_margin_pass,
        "positive_control_detector_ok": positive_ok,
        "spontaneous_heatkernel_fractal_memory_claim": claim,
        "important_scope_note": "This heat-kernel audit is a stricter follow-up to the Z-residual test. H1 receives no log-periodic injection. A positive result would require specificity against H0/static/nonlattice/smooth-spectrum/surrogate controls.",
    }


def make_plots(curves_df: pd.DataFrame, period_df: pd.DataFrame, summary_df: pd.DataFrame, out: Path, cfg: HeatKernelMemoryConfig):
    if plt is None:
        return []
    made = []
    prefix = "ROSG_Test_H1_HeatKernel_FractalMemory_Audit"
    # Residual comparison at activation_center.
    fig, ax = plt.subplots(figsize=(9, 5))
    cases_show = ["H1_Z_coupled", "H0_planar", "CTRL_H1_static_full_channel", "CTRL_smooth_Weyl_spectrum_matched_count"]
    for case in cases_show:
        d = curves_df[(curves_df["case"] == case) & (curves_df["target_i"] == max(cfg.levels)) & (curves_df["Z_slice_label"] == "activation_center")]
        if len(d):
            ax.plot(d["logu"], d["heat_residual_logP_std"], label=case)
    ax.axhline(0, linewidth=0.8)
    ax.set_title("Heat-trace residuals in log u at activation center (highest level)")
    ax.set_xlabel("log u")
    ax.set_ylabel("standardized residual of log P")
    ax.legend(fontsize=8)
    fig.tight_layout()
    p = out / f"{prefix}_heat_residuals_logu.png"; fig.savefig(p, dpi=160); plt.close(fig); made.append(str(p))
    # Power/FAP summary.
    fig, ax = plt.subplots(figsize=(10, 5))
    s = summary_df[summary_df["Z_slice_label"].isin(["activation_center", "post_saturation"])].copy()
    labels = (s["case"] + "\n" + s["Z_slice_label"]).tolist()
    x = np.arange(len(s))
    ax.bar(x, s["median_best_power"].to_numpy(float))
    ax.axhline(cfg.min_power_threshold, linestyle="--", linewidth=1.0, label="min power")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=7)
    ax.set_ylabel("median best sinusoid power")
    ax.set_title("Heat-kernel logu periodicity power by case")
    ax.legend(fontsize=8)
    fig.tight_layout()
    p = out / f"{prefix}_power_summary.png"; fig.savefig(p, dpi=160); plt.close(fig); made.append(str(p))
    # Best periods.
    fig, ax = plt.subplots(figsize=(10, 5))
    for case, d in period_df[period_df["Z_slice_label"] == "activation_center"].groupby("case"):
        ax.plot(d["target_i"], d["best_period_logu"], marker="o", label=case)
    ax.axhline(math.log(4.0), linestyle="--", linewidth=1.0, label="ln 4 reference")
    ax.set_xlabel("level i")
    ax.set_ylabel("best period in log u")
    ax.set_title("Blind best periods by level, activation-center slice")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    p = out / f"{prefix}_best_periods_logu.png"; fig.savefig(p, dpi=160); plt.close(fig); made.append(str(p))
    # Verdict flags.
    report_like = global_verdict(summary_df, cfg)
    flags = {k:v for k,v in report_like.items() if isinstance(v, bool)}
    fig, ax = plt.subplots(figsize=(8, 3.5))
    names = list(flags.keys()); vals = [1 if flags[n] else 0 for n in names]
    ax.bar(np.arange(len(names)), vals)
    ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names, rotation=35, ha="right", fontsize=8)
    ax.set_ylim(0, 1.2); ax.set_yticks([0,1]); ax.set_yticklabels(["False","True"])
    ax.set_title("Heat-kernel fractal memory verdict flags")
    fig.tight_layout()
    p = out / f"{prefix}_verdict_flags.png"; fig.savefig(p, dpi=160); plt.close(fig); made.append(str(p))
    return made


def run_heatkernel_fractal_memory_audit(cfg: HeatKernelMemoryConfig):
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(cfg.random_seed)
    base_cases = [
        "H1_Z_coupled",
        "H0_planar",
        "CTRL_H1_static_full_channel",
        "CTRL_nonlattice_Z_centers",
        "CTRL_smooth_Weyl_spectrum_matched_count",
        "CTRL_dropout_Z_effective_shift",
    ]
    all_curves, all_periods = [], []
    for case in base_cases:
        curves, periods = compute_curve_records_for_case(cfg, case, rng)
        all_curves.extend(curves); all_periods.extend(periods)
    curves_df = pd.DataFrame(all_curves)
    period_df = pd.DataFrame(all_periods)
    pos_curves, pos_periods = add_positive_control_from_h1(curves_df, cfg, rng)
    surr_curves, surr_periods = make_residual_surrogates(curves_df, period_df, cfg, rng)
    curves_df = pd.concat([curves_df, pos_curves, surr_curves], ignore_index=True)
    period_df = pd.concat([period_df, pos_periods, surr_periods], ignore_index=True)
    summary_df = summarize_periodicity(period_df, cfg)
    report = {
        "status": "completed",
        "test_name": "ROSG_Test_H1_HeatKernel_FractalMemory_Audit",
        "config": asdict(cfg),
        "report": global_verdict(summary_df, cfg),
    }
    prefix = "ROSG_Test_H1_HeatKernel_FractalMemory_Audit"
    curves_path = out / f"{prefix}_curves.csv"
    period_path = out / f"{prefix}_periodicity.csv"
    summary_path = out / f"{prefix}_summary.csv"
    report_path = out / f"{prefix}_report.json"
    curves_df.to_csv(curves_path, index=False)
    period_df.to_csv(period_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    plot_paths = make_plots(curves_df, period_df, summary_df, out, cfg)
    package_path = Path("/mnt/data/ROSG_Test_H1_HeatKernel_FractalMemory_Audit_package.zip")
    with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [curves_path, period_path, summary_path, report_path] + [Path(x) for x in plot_paths]:
            if Path(p).exists():
                zf.write(p, arcname=Path(p).name)
        script_path = Path("/mnt/data/ROSG_Test_H1_HeatKernel_FractalMemory_Audit.py")
        if script_path.exists():
            zf.write(script_path, arcname=script_path.name)
        nb_path = Path("/mnt/data/ROSG_Test_H1_HeatKernel_FractalMemory_Audit.ipynb")
        if nb_path.exists():
            zf.write(nb_path, arcname=nb_path.name)
    return report, curves_df, period_df, summary_df, str(package_path)


if __name__ == "__main__":
    cfg = HeatKernelMemoryConfig()
    report, curves, period, summary, package_path = run_heatkernel_fractal_memory_audit(cfg)
    print(json.dumps({
        "status": report["status"],
        "test_name": report["test_name"],
        "verdict": report["report"]["heatkernel_fractal_memory_verdict"],
        "h1_meaningful_slices_coherent_pass": report["report"]["h1_meaningful_slices_coherent_pass"],
        "null_controls_rejected": report["report"]["null_controls_rejected"],
        "null_controls_or_surrogates_coherent": report["report"]["null_controls_or_surrogates_coherent"],
        "h1_power_margin_over_max_null": report["report"]["h1_power_margin_over_max_null"],
        "positive_control_detector_ok": report["report"]["positive_control_detector_ok"],
        "spontaneous_heatkernel_fractal_memory_claim": report["report"]["spontaneous_heatkernel_fractal_memory_claim"],
        "package": package_path,
    }, indent=2))


{
  "status": "completed",
  "test_name": "ROSG_Test_H1_HeatKernel_FractalMemory_Audit",
  "verdict": "H1_heatkernel_no_coherent_fractal_memory_detected",
  "h1_meaningful_slices_coherent_pass": false,
  "null_controls_rejected": false,
  "null_controls_or_surrogates_coherent": [
    "CTRL_circular_shift_heat_residual",
    "CTRL_nonlattice_Z_centers",
    "CTRL_phase_surrogate_heat_residual"
  ],
  "h1_power_margin_over_max_null": 0.430846403446907,
  "positive_control_detector_ok": true,
  "spontaneous_heatkernel_fractal_memory_claim": false,
  "package": "/mnt/data/ROSG_Test_H1_HeatKernel_FractalMemory_Audit_package.zip"
}


## Synthèse des Résultats

**Verdict du test de mémoire fractale du noyau de chaleur :** `H1_heatkernel_no_coherent_fractal_memory_detected`

**Passage cohérent des tranches significatives H1 :** `false`

**Contrôles nuls rejetés :** `false`

**Contrôles nuls ou surrogates cohérents :** `['CTRL_circular_shift_heat_residual', 'CTRL_nonlattice_Z_centers', 'CTRL_phase_surrogate_heat_residual']`

**Marge de puissance H1 sur le maximum nul :** `0.430846403446907`

**Contrôle positif du détecteur OK :** `true`

**Revendication spontanée de la mémoire fractale du noyau de chaleur :** `false`

En résumé, le test n'a pas détecté de mémoire fractale cohérente du noyau de chaleur dans le canal H1, et les contrôles nuls n'ont pas été rejetés, ce qui indique que la périodicité observée n'est pas spécifique.